In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [7]:
api_key = os.getenv("GOOGLE_API_KEY")

if api_key:
    print("✅ API key loaded")
    print(f"Length: {len(api_key)}")
else:
    print("❌ API key not found")

✅ API key loaded
Length: 53


In [4]:
# Create Gemini client
from google import genai
from langsmith import traceable

import os



# Gemini Client
client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

@traceable(name="gemini_call")
def get_gemini_response(user_input):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_input
    )

    return response.text


personality = "You are a helpful assistant that provides concise and accurate answers to user questions."
message = [
    "role: system",
    f"content: {personality}"
]



print("🤖 Gemini Chatbot")
print("Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "quit":
        print("Bot: Bye!")
        break

    try:
        reply = get_gemini_response(user_input)

        print("Bot:", reply)

    except Exception as e:
        print(f"❌ Error: {e}")

🤖 Gemini Chatbot
Type 'quit' to exit.

Bot: Hello! I'm doing well, thank you for asking. I'm ready and here to help you.

How can I assist you today?
Bot: Bye!


In [10]:
!pip install langchain google-genai 

  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached websockets-15.0.1-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/554.9 kB ? eta -:--:--
   ---------------------------------------- 554.9/554.9 kB 9.5 MB/s  0:00:00
Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl (56 kB)
Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl (41 kB)
Using cached httpx-0.28.1-py3-none-any

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # Dummy implementation for weather tool
    return f"The current weather in {location} is sunny."

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# create_react_agent from langgraph handles execution directly
# If we are using creat_agent from langchain, then result works with dictionary and key is message 
agent_executor = create_react_agent(
    model=llm,
    tools=[get_weather],  # add your tools here,
    prompt="You are a helpful assistant that provides concise and accurate answers to user questions."
)

# To run it:
result = agent_executor.invoke({"messages": [("user", "your query here")]})

C:\Users\shiva\AppData\Local\Temp\ipykernel_9428\3413455239.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [25]:
#Run the agent with a user query
result = agent_executor.invoke({"messages": [("user", "What's the weather like in New York?")]})
print(result)

{'messages': [HumanMessage(content="What's the weather like in New York?", additional_kwargs={}, response_metadata={}, id='5fc37fff-0330-4a1a-be7b-3f1a69eb56ff'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'48ab8772-cb4e-461e-a598-ef3504baeffb': 'CosCAQw51scrERcNaSL3mczGrE9y2JN0vyR2Axhz3dPLTEeyc1f/X8lBy/txNrAiiCddTn6LKiTDF64kM30bBoQvu5j6TbQFM1NfuvKHR9wnU1MbWrT0CaLwqGbbPCHOuv96RbXqGjVDBynQEeK1vc73uKvtPmddPRL71fnSxsfZnkYuxDIVKn9j1zU+gruRiNhQiFpbYSbegeDJaU014gLmScv7fcq0bmM/p4708IT0CXtLn+wrmMPBI8ZraVJIsp6NW9wwcsCchCnMmICjbbW39ud8uWWxkJFDn99H3eqa56tcgDLnrYjaQ20dPT0mJsWbhf+E9zYhZCFfwqBn8sa6pZnshqBRifISyyqL'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ebfca-aab8-7352-a2af-4cbcdde1ee3e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, '

In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

# ── Tool 1: Calculator ─────────────────────────────────────────
@tool
def calculator(expression: str) -> str:
    """Evaluates a math expression. Example: '2 + 2', '10 * 5', '100 / 4'."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

# ── Tool 2: Word Counter ───────────────────────────────────────
@tool
def word_counter(text: str) -> str:
    """Counts words, characters, and sentences in a given text."""
    words = len(text.split())
    characters = len(text)
    characters_no_spaces = len(text.replace(" ", ""))
    sentences = text.count('.') + text.count('!') + text.count('?')
    return (
        f"Words: {words}\n"
        f"Characters (with spaces): {characters}\n"
        f"Characters (no spaces): {characters_no_spaces}\n"
        f"Sentences: {sentences}"
    )

# ── LLM Setup ─────────────────────────────────────────────────
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# ── Agent ─────────────────────────────────────────────────────
agent = create_react_agent(
    model=llm,
    tools=[calculator, word_counter],
    prompt="You are a helpful assistant. Use the calculator tool for math and the word_counter tool for text analysis."
)

# ── Chat Loop ─────────────────────────────────────────────────
print("Bot ready! Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        print("Goodbye!")
        break

    result = agent.invoke({
        "messages": [("user", user_input)]
    })

    print(f"Bot: {result['messages'][-1].content}\n")

Bot ready! Type 'exit' to quit.



C:\Users\shiva\AppData\Local\Temp\ipykernel_9428\1780852834.py:37: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Bot: [{'type': 'text', 'text': 'As an AI, I cannot directly "make you happy." My purpose is to assist you with tasks like answering questions, providing information, or performing calculations and text analysis.\n\nIs there something specific I can help you with that might brighten your day, like:\n\n*   **Answering a question you have?**\n*   **Helping you with a writing task?**\n*   **Performing a calculation?**\n*   **Telling you a fun fact?**', 'extras': {'signature': 'CqsDAQw51sf9/DC/So2v0q/wNNHLJ+JJEsrVqbrpAcS8UAbzvVKZOPxZYipIBMBb4uj8egcDhZcMSP4+RRufi0P+6REC+0ZwM+oJVRLysHMV9VnhvxVZhA3Vhd4xRbX60ArACcPvOJjme9N+q3T8dKnHTx1l71wHrDLkLzA6MlCzw3YJWT88Dyq6pRnnZ+HowGQBfLOsmDjzDZ656q+Fl9an/q9gY5DRczxGkUaLUctBnJk7jcrCoTVE1Pgvp8k7kxCWdnyp/5VmMT2pf5KtdhAYdXy+wsu9t5jEGZ31WX7Mv25SjCTsou5mVTIoXVx8u7GvThmw7j5rzcgQoVrnBFAnXxMlSpP0aaPzMmQTxJinOqEBub1RySl04LVLB6sTmdphpUSNeAIJ9Xf7rNWb/46p/hMcOifQIMBAkj6XQsfGr9MAUW7xAu6dwZd8RgFljvxOH+Z0hK4gKzc2cZPptZRYxTG+lDWBxYfLqzxlVVInU2CBrSNMKb0QLhQtSoWD0hYm0Yu7oJ